# 🔬 AISEHack Phase 2 — EDA Part 2
**Targeting the 3 critical gaps identified after Part 1:**
1. SAR backscatter distributions: Flood vs WaterBody (pixel-level)
2. Train vs Val class composition — are splits balanced?
3. Spatial adjacency between flood and waterbody pixels
4. BONUS: SAR texture features — do they separate classes?
5. BONUS: Patch-level difficulty map — which patches will hurt most?

In [ ]:
# ── Re-run setup (paste from Part 1 or run after Part 1 kernel) ───────────────
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import ndimage
from scipy.stats import ks_2samp
import rasterio
warnings.filterwarnings('ignore')

BASE      = '/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data'
IMG_DIR   = os.path.join(BASE, 'image')
LBL_DIR   = os.path.join(BASE, 'label')
PRED_DIR  = os.path.join(BASE, 'prediction', 'image')
SPLIT_DIR = os.path.join(BASE, 'split')

def load_split(fname):
    with open(os.path.join(SPLIT_DIR, fname)) as f:
        return [l.strip() for l in f if l.strip()]

train_ids = load_split('train.txt')
val_ids   = load_split('val.txt')
test_ids  = load_split('test.txt')

all_images = sorted(glob.glob(os.path.join(IMG_DIR,  '*.tif')))
all_labels = sorted(glob.glob(os.path.join(LBL_DIR,  '*.tif')))
all_preds  = sorted(glob.glob(os.path.join(PRED_DIR, '*.tif')))

def build_path_map(file_list):
    d = {}
    for f in file_list:
        key = os.path.basename(f).replace('_image.tif','').replace('_label.tif','').replace('.tif','')
        d[key] = f
    return d

img_map  = build_path_map(all_images)
lbl_map  = build_path_map(all_labels)
pred_map = build_path_map(all_preds)

# Detect band layout
with rasterio.open(all_images[0]) as src:
    n_bands = src.count

if n_bands == 7:
    idx_HH, idx_HV = 1, 2
    idx_G, idx_R, idx_NIR, idx_SWIR = 3, 4, 5, 6
    CHANNEL_NAMES = ['FloodLabel','SAR_HH','SAR_HV','Green','Red','NIR','SWIR']
else:
    idx_HH, idx_HV = 0, 1
    idx_G, idx_R, idx_NIR, idx_SWIR = 2, 3, 4, 5
    CHANNEL_NAMES = ['SAR_HH','SAR_HV','Green','Red','NIR','SWIR']

CLASS_NAMES  = {0:'NoFlood', 1:'Flood', 2:'WaterBody'}
CLASS_COLORS = {0:'#a8d5a2', 1:'#e63946', 2:'#457b9d'}

def load_pair(pid):
    img_path = lbl_path = None
    for k, v in img_map.items():
        if pid in k or k in pid: img_path = v; break
    for k, v in lbl_map.items():
        if pid in k or k in pid: lbl_path = v; break
    if not img_path or not lbl_path: return None, None
    with rasterio.open(img_path) as s: img = s.read().astype(np.float32)
    with rasterio.open(lbl_path) as s: lbl = s.read(1)
    return img, lbl

labelled_ids = train_ids + val_ids
print(f'✅ Setup complete — {len(labelled_ids)} labelled patches, {n_bands} bands')

---
## GAP 1: SAR Backscatter — Flood vs WaterBody at Pixel Level
**Question: Do SAR HH and HV pixel values actually differ between flood and waterbody?**
This directly determines whether a model can separate them spectrally or must rely purely on spatial context.

In [ ]:
# ── Collect raw SAR pixel values per class ────────────────────────────────────
MAX_SAMPLES = 20000  # per class
sar_hh_by_class = {0:[], 1:[], 2:[]}
sar_hv_by_class = {0:[], 1:[], 2:[]}

rng = np.random.default_rng(42)

for pid in labelled_ids:
    img, lbl = load_pair(pid)
    if img is None: continue
    hh = img[idx_HH]
    hv = img[idx_HV]
    for cls in [0, 1, 2]:
        mask = (lbl == cls)
        if mask.sum() == 0: continue
        ridx, cidx = np.where(mask)
        if len(ridx) > 1000:  # sample per patch
            sel = rng.choice(len(ridx), 1000, replace=False)
            ridx, cidx = ridx[sel], cidx[sel]
        sar_hh_by_class[cls].extend(hh[ridx, cidx].tolist())
        sar_hv_by_class[cls].extend(hv[ridx, cidx].tolist())

# Cap to MAX_SAMPLES
for cls in [0,1,2]:
    if len(sar_hh_by_class[cls]) > MAX_SAMPLES:
        sel = rng.choice(len(sar_hh_by_class[cls]), MAX_SAMPLES, replace=False)
        sar_hh_by_class[cls] = np.array(sar_hh_by_class[cls])[sel]
        sar_hv_by_class[cls] = np.array(sar_hv_by_class[cls])[sel]
    else:
        sar_hh_by_class[cls] = np.array(sar_hh_by_class[cls])
        sar_hv_by_class[cls] = np.array(sar_hv_by_class[cls])

print('Samples collected per class:')
for cls in [0,1,2]:
    print(f'  Class {cls} ({CLASS_NAMES[cls]}): {len(sar_hh_by_class[cls]):,}')

In [ ]:
# ── SAR distributions: KDE + Box plots ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
fig.suptitle('SAR Backscatter: Flood vs WaterBody — The Core Separability Question',
             fontsize=13, fontweight='bold')

# Clip range for display
clip_lo, clip_hi = np.percentile(
    np.concatenate([sar_hh_by_class[c] for c in [0,1,2]]), [1, 99])

for col, (band_name, band_data) in enumerate([
    ('SAR HH', sar_hh_by_class),
    ('SAR HV', sar_hv_by_class)
]):
    # Row 0: Overlapping histograms
    ax = axes[0][col]
    for cls in [0, 1, 2]:
        vals = np.clip(band_data[cls], clip_lo, clip_hi)
        ax.hist(vals, bins=80, alpha=0.55, density=True,
                color=CLASS_COLORS[cls], label=CLASS_NAMES[cls], edgecolor='none')
    ax.set_title(f'{band_name} — Overlapping Density')
    ax.set_xlabel(f'{band_name} value')
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(alpha=0.3)

    # Row 1: Box plots
    ax = axes[1][col]
    bp = ax.boxplot(
        [np.clip(band_data[c], clip_lo, clip_hi) for c in [0,1,2]],
        patch_artist=True,
        medianprops={'color':'black','linewidth':2},
        showfliers=False
    )
    for patch, cls in zip(bp['boxes'], [0,1,2]):
        patch.set_facecolor(CLASS_COLORS[cls])
        patch.set_alpha(0.8)
    ax.set_xticklabels([CLASS_NAMES[c] for c in [0,1,2]])
    ax.set_title(f'{band_name} — Box Plot')
    ax.set_ylabel(f'{band_name} value')
    ax.grid(axis='y', alpha=0.3)

# Row 0, col 2: SAR HH vs HV scatter — Flood vs WaterBody only
ax = axes[0][2]
for cls in [1, 2]:  # flood and waterbody only
    sel = rng.choice(len(sar_hh_by_class[cls]),
                     min(2000, len(sar_hh_by_class[cls])), replace=False)
    ax.scatter(
        np.clip(sar_hh_by_class[cls][sel], clip_lo, clip_hi),
        np.clip(sar_hv_by_class[cls][sel], clip_lo, clip_hi),
        alpha=0.3, s=8, color=CLASS_COLORS[cls], label=CLASS_NAMES[cls]
    )
ax.set_xlabel('SAR HH')
ax.set_ylabel('SAR HV')
ax.set_title('Flood vs WaterBody — HH vs HV scatter')
ax.legend()
ax.grid(alpha=0.3)

# Row 1, col 2: KS test result summary
ax = axes[1][2]
ax.axis('off')
results = []
for band_name, band_data in [('SAR_HH', sar_hh_by_class), ('SAR_HV', sar_hv_by_class)]:
    for (c1, c2) in [(0,1), (0,2), (1,2)]:
        stat, p = ks_2samp(band_data[c1], band_data[c2])
        results.append({
            'Band': band_name,
            'Comparison': f'{CLASS_NAMES[c1]} vs {CLASS_NAMES[c2]}',
            'KS stat': round(stat, 4),
            'p-value': f'{p:.2e}',
            'Separable?': '✅ YES' if stat > 0.1 else '⚠️ WEAK' if stat > 0.05 else '❌ NO'
        })

res_df = pd.DataFrame(results)
ax.text(0.05, 0.95, 'Kolmogorov-Smirnov Test\n(KS stat > 0.1 = meaningfully separable)',
        transform=ax.transAxes, va='top', fontsize=10, fontweight='bold')
table = ax.table(cellText=res_df.values, colLabels=res_df.columns,
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.6)

plt.tight_layout()
plt.savefig('sar_separability.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nKEY NUMBERS:')
for cls in [1, 2]:
    print(f'  {CLASS_NAMES[cls]} — HH mean: {sar_hh_by_class[cls].mean():.4f}  '
          f'HV mean: {sar_hv_by_class[cls].mean():.4f}')
hh_ks, _ = ks_2samp(sar_hh_by_class[1], sar_hh_by_class[2])
hv_ks, _ = ks_2samp(sar_hv_by_class[1], sar_hv_by_class[2])
print(f'\n  Flood vs WaterBody KS — HH: {hh_ks:.4f}  HV: {hv_ks:.4f}')
print(f'  (Interpretation: {"Spectrally separable!" if hh_ks > 0.1 else "Cannot separate spectrally — model must use spatial context"})')

---
## GAP 2: Train vs Val Split — Class Composition
**Question: Are the splits balanced? Does val contain rare high-waterbody patches that train never saw?**

In [ ]:
# ── Per-patch class stats for train and val separately ────────────────────────
def compute_patch_stats(pid_list, split_name):
    rows = []
    for pid in pid_list:
        img, lbl = load_pair(pid)
        if lbl is None: continue
        total = lbl.size
        row = {'split': split_name, 'patch_id': pid}
        for cls in [0,1,2]:
            cnt = int((lbl == cls).sum())
            row[CLASS_NAMES[cls] + '_pct'] = round(cnt / total * 100, 2)
            row[CLASS_NAMES[cls] + '_cnt'] = cnt
        row['has_flood']     = row['Flood_cnt'] > 0
        row['has_waterbody'] = row['WaterBody_cnt'] > 0
        rows.append(row)
    return pd.DataFrame(rows)

train_df = compute_patch_stats(train_ids, 'train')
val_df   = compute_patch_stats(val_ids,   'val')
all_df   = pd.concat([train_df, val_df], ignore_index=True)

print('═' * 60)
print('  TRAIN vs VAL — CLASS PERCENTAGE SUMMARY')
print('═' * 60)
for split_name, df in [('TRAIN', train_df), ('VAL', val_df)]:
    print(f'\n  [{split_name}]  n_patches={len(df)}')
    for cls_name in ['NoFlood_pct', 'Flood_pct', 'WaterBody_pct']:
        vals = df[cls_name]
        print(f'    {cls_name:<18}: mean={vals.mean():.1f}%  '
              f'min={vals.min():.1f}%  max={vals.max():.1f}%  '
              f'std={vals.std():.1f}%')
    print(f'    Patches with flood      : {df["has_flood"].sum()} / {len(df)}')
    print(f'    Patches with waterbody  : {df["has_waterbody"].sum()} / {len(df)}')

In [ ]:
# ── Visual comparison of train vs val distributions ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Train vs Val — Class Distribution Comparison\n'
             '(If val has patches train never saw → model will fail there)',
             fontsize=12, fontweight='bold')

for ax, col, title, color_pair in zip(
    axes,
    ['Flood_pct', 'WaterBody_pct', 'NoFlood_pct'],
    ['Flood %', 'WaterBody %', 'NoFlood %'],
    [('#e63946','#c77d7d'), ('#457b9d','#7ba8c4'), ('#a8d5a2','#c8e8c5')]
):
    bins = np.linspace(
        min(train_df[col].min(), val_df[col].min()),
        max(train_df[col].max(), val_df[col].max()),
        20
    )
    ax.hist(train_df[col], bins=bins, alpha=0.7, color=color_pair[0],
            label=f'Train (n={len(train_df)})', edgecolor='white')
    ax.hist(val_df[col],   bins=bins, alpha=0.7, color=color_pair[1],
            label=f'Val   (n={len(val_df)})',   edgecolor='white')
    ax.axvline(train_df[col].mean(), color=color_pair[0], linestyle='--', linewidth=2)
    ax.axvline(val_df[col].mean(),   color=color_pair[1], linestyle='--', linewidth=2)
    ax.set_xlabel(title)
    ax.set_ylabel('Number of patches')
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('train_val_split.png', dpi=120, bbox_inches='tight')
plt.show()

# Flag dangerous patches — val patches with class % far from train distribution
print('\n⚠️  VAL PATCHES OUTSIDE TRAIN DISTRIBUTION (potential hard cases):')
for col, cls_name in [('Flood_pct','Flood'), ('WaterBody_pct','WaterBody')]:
    train_max = train_df[col].max()
    danger = val_df[val_df[col] > train_max]
    if len(danger) > 0:
        print(f'  {cls_name}: {len(danger)} val patches exceed train max ({train_max:.1f}%)')
        print(danger[['patch_id', col]].to_string(index=False))
    else:
        print(f'  {cls_name}: ✅ Val within train distribution range')

In [ ]:
# ── Scatter plot: each patch as a dot (flood% vs waterbody%) ─────────────────
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(train_df['Flood_pct'], train_df['WaterBody_pct'],
           c='#e63946', s=80, alpha=0.8, label='Train', zorder=3, edgecolors='white')
ax.scatter(val_df['Flood_pct'], val_df['WaterBody_pct'],
           c='#457b9d', s=80, alpha=0.8, label='Val',   zorder=3,
           marker='D', edgecolors='white')

# Annotate outlier patches
for _, row in all_df.iterrows():
    if row['Flood_pct'] > 30 or row['WaterBody_pct'] > 40:
        ax.annotate(row['patch_id'].split('_pid_')[-1],
                    (row['Flood_pct'], row['WaterBody_pct']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Flood pixel %', fontsize=12)
ax.set_ylabel('WaterBody pixel %', fontsize=12)
ax.set_title('Each Patch: Flood% vs WaterBody%\n'
             '(Train=red circles, Val=blue diamonds — gaps = distribution mismatch)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('patch_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

---
## GAP 3: Spatial Adjacency — How Much Do Flood and WaterBody Touch?
**Question: Are flood pixels mostly at the boundary of waterbodies, or spatially separate?**
This determines how hard the segmentation boundary prediction is.

In [ ]:
# ── Adjacency analysis ────────────────────────────────────────────────────────
# For each flood pixel, check if any of its 8 neighbors is a waterbody pixel
# This tells us: what fraction of flood pixels are at the flood-waterbody interface?

from scipy.ndimage import binary_dilation

struct = np.ones((3, 3), dtype=bool)  # 8-connectivity

adjacency_stats = []

for pid in labelled_ids:
    img, lbl = load_pair(pid)
    if lbl is None: continue

    flood_mask = (lbl == 1)
    water_mask = (lbl == 2)

    if flood_mask.sum() == 0 or water_mask.sum() == 0:
        adjacency_stats.append({
            'patch_id': pid,
            'flood_px': int(flood_mask.sum()),
            'water_px': int(water_mask.sum()),
            'flood_adj_to_water_pct': 0.0,
            'water_adj_to_flood_pct': 0.0,
            'both_present': False
        })
        continue

    # Dilate flood mask → which pixels border flood?
    dilated_flood = binary_dilation(flood_mask, structure=struct)
    dilated_water = binary_dilation(water_mask, structure=struct)

    # Flood pixels adjacent to waterbody
    flood_touching_water = flood_mask & dilated_water
    # Waterbody pixels adjacent to flood
    water_touching_flood = water_mask & dilated_flood

    adjacency_stats.append({
        'patch_id': pid,
        'flood_px': int(flood_mask.sum()),
        'water_px': int(water_mask.sum()),
        'flood_adj_to_water_pct': float(flood_touching_water.sum() / flood_mask.sum() * 100),
        'water_adj_to_flood_pct': float(water_touching_flood.sum() / water_mask.sum() * 100),
        'both_present': True
    })

adj_df = pd.DataFrame(adjacency_stats)
both = adj_df[adj_df['both_present']]

print('═' * 60)
print('  FLOOD ↔ WATERBODY SPATIAL ADJACENCY')
print('═' * 60)
print(f'  Patches with BOTH flood and waterbody : {len(both)} / {len(adj_df)}')
print(f'\n  Of those {len(both)} patches:')
print(f'  Flood pixels touching WaterBody  — mean: {both["flood_adj_to_water_pct"].mean():.1f}%')
print(f'                                     max:  {both["flood_adj_to_water_pct"].max():.1f}%')
print(f'                                     min:  {both["flood_adj_to_water_pct"].min():.1f}%')
print(f'\n  WaterBody pixels touching Flood  — mean: {both["water_adj_to_flood_pct"].mean():.1f}%')
print(f'                                     max:  {both["water_adj_to_flood_pct"].max():.1f}%')
print(f'                                     min:  {both["water_adj_to_flood_pct"].min():.1f}%')

In [ ]:
# ── Visualise the boundary zone in the worst-case patches ────────────────────
# Pick patches where flood-waterbody adjacency is highest
worst_patches = both.nlargest(3, 'flood_adj_to_water_pct')['patch_id'].values

fig, axes = plt.subplots(len(worst_patches), 4, figsize=(20, 5 * len(worst_patches)))
if len(worst_patches) == 1: axes = [axes]

legend_patches = [
    mpatches.Patch(color='#a8d5a2', label='0 NoFlood'),
    mpatches.Patch(color='#e63946', label='1 Flood'),
    mpatches.Patch(color='#457b9d', label='2 WaterBody'),
    mpatches.Patch(color='#ff9f00', label='Flood∩WB boundary'),
]

def norm(b):
    p2,p98 = np.percentile(b,2), np.percentile(b,98)
    return np.clip((b-p2)/(p98-p2+1e-8),0,1)

def make_rgb(lbl):
    h,w = lbl.shape; rgb = np.zeros((h,w,3))
    rgb[lbl==0]=[0.659,0.835,0.635]; rgb[lbl==1]=[0.902,0.224,0.275]; rgb[lbl==2]=[0.271,0.482,0.616]
    return rgb

for row_i, pid in enumerate(worst_patches):
    img, lbl = load_pair(pid)
    stat = adj_df[adj_df['patch_id'] == pid].iloc[0]

    flood_mask = (lbl == 1)
    water_mask = (lbl == 2)
    dil_water  = binary_dilation(water_mask, structure=struct)
    boundary   = flood_mask & dil_water  # flood pixels touching waterbody

    axes[row_i][0].set_ylabel(
        f'Patch {pid.split("_pid_")[-1]}\n'
        f'{stat["flood_adj_to_water_pct"]:.0f}% flood\ntouches WB', fontsize=8)

    # SAR HH
    axes[row_i][0].imshow(norm(img[idx_HH]), cmap='gray')
    axes[row_i][0].set_title('SAR HH')

    # SAR HV
    axes[row_i][1].imshow(norm(img[idx_HV]), cmap='gray')
    axes[row_i][1].set_title('SAR HV')

    # Label
    axes[row_i][2].imshow(make_rgb(lbl))
    axes[row_i][2].set_title('Ground Truth (3-class)')
    axes[row_i][2].legend(handles=legend_patches[:3], loc='lower right', fontsize=7)

    # Boundary zone overlay
    overlay = make_rgb(lbl).copy()
    overlay[boundary] = [1.0, 0.624, 0.0]  # orange = boundary zone
    axes[row_i][3].imshow(overlay)
    axes[row_i][3].set_title('Flood-WB Boundary (orange)')
    axes[row_i][3].legend(handles=legend_patches, loc='lower right', fontsize=7)

    for ax in axes[row_i]: ax.axis('off')

plt.suptitle('Worst-Case Patches: Where Flood Meets WaterBody\n'
             '(Orange = pixels model must correctly classify despite being at the border)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('boundary_analysis.png', dpi=110, bbox_inches='tight')
plt.show()

---
## BONUS: SAR Texture Features
**Since raw SAR values may not separate flood from waterbody, do local texture statistics?**
Local std/variance in a small window captures surface roughness — potentially discriminative.

In [ ]:
# ── Local SAR texture (3x3 and 7x7 std) per class ────────────────────────────
from scipy.ndimage import uniform_filter

def local_std(arr, radius=3):
    """Fast local standard deviation via uniform filter."""
    mean  = uniform_filter(arr.astype(np.float64), size=radius)
    mean2 = uniform_filter(arr.astype(np.float64)**2, size=radius)
    var   = np.maximum(mean2 - mean**2, 0)
    return np.sqrt(var)

texture_by_class = {cls: {'hh_std3':[], 'hh_std7':[], 'hv_std3':[], 'hv_std7':[]}
                    for cls in [0,1,2]}

for pid in labelled_ids[:25]:  # 25 patches enough
    img, lbl = load_pair(pid)
    if img is None: continue

    hh_s3 = local_std(img[idx_HH], 3)
    hh_s7 = local_std(img[idx_HH], 7)
    hv_s3 = local_std(img[idx_HV], 3)
    hv_s7 = local_std(img[idx_HV], 7)

    for cls in [0,1,2]:
        mask = (lbl == cls)
        if mask.sum() < 10: continue
        ridx, cidx = np.where(mask)
        if len(ridx) > 1500:
            sel = rng.choice(len(ridx), 1500, replace=False)
            ridx, cidx = ridx[sel], cidx[sel]
        texture_by_class[cls]['hh_std3'].extend(hh_s3[ridx,cidx].tolist())
        texture_by_class[cls]['hh_std7'].extend(hh_s7[ridx,cidx].tolist())
        texture_by_class[cls]['hv_std3'].extend(hv_s3[ridx,cidx].tolist())
        texture_by_class[cls]['hv_std7'].extend(hv_s7[ridx,cidx].tolist())

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('SAR LOCAL TEXTURE (std in window) by Class\n'
             'Key insight: Open water = low texture. Flood on vegetation = higher texture.',
             fontsize=12, fontweight='bold')

for ax, feat, title in zip(
    axes,
    ['hh_std3', 'hh_std7', 'hv_std3', 'hv_std7'],
    ['HH std (3x3)', 'HH std (7x7)', 'HV std (3x3)', 'HV std (7x7)']
):
    data = []
    labels_plot = []
    clip_hi = np.percentile(np.concatenate([texture_by_class[c][feat] for c in [0,1,2]]), 99)
    for cls in [0,1,2]:
        vals = np.clip(texture_by_class[cls][feat], 0, clip_hi)
        if len(vals) > 5000: vals = rng.choice(vals, 5000, replace=False)
        data.append(vals)
        labels_plot.append(CLASS_NAMES[cls])

    bp = ax.boxplot(data, patch_artist=True,
                    medianprops={'color':'black','linewidth':2}, showfliers=False)
    for patch, cls in zip(bp['boxes'], [0,1,2]):
        patch.set_facecolor(CLASS_COLORS[cls]); patch.set_alpha(0.8)
    ax.set_xticklabels(labels_plot)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)

    # KS test Flood vs WaterBody
    ks, _ = ks_2samp(texture_by_class[1][feat], texture_by_class[2][feat])
    ax.text(0.5, 0.97, f'F vs WB KS={ks:.3f}',
            transform=ax.transAxes, ha='center', va='top', fontsize=9,
            color='green' if ks > 0.1 else 'red',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('sar_texture.png', dpi=120, bbox_inches='tight')
plt.show()

print('KS stats (Flood vs WaterBody) for texture features:')
for feat in ['hh_std3','hh_std7','hv_std3','hv_std7']:
    ks, _ = ks_2samp(texture_by_class[1][feat], texture_by_class[2][feat])
    sep = '✅ SEPARABLE' if ks > 0.1 else '⚠️ WEAK' if ks > 0.05 else '❌ NOT SEPARABLE'
    print(f'  {feat:<12}: KS={ks:.4f}  →  {sep}')

---
## BONUS: Patch Difficulty Map
**Which patches will be hardest to predict? Rank them by difficulty score.**

In [ ]:
# ── Difficulty = patches where flood and waterbody both exist AND touch ───────
diff_rows = []

for pid in labelled_ids:
    img, lbl = load_pair(pid)
    if lbl is None: continue

    flood_mask = (lbl == 1)
    water_mask = (lbl == 2)
    total      = lbl.size

    flood_pct = flood_mask.sum() / total * 100
    water_pct = water_mask.sum() / total * 100

    if flood_mask.sum() > 0 and water_mask.sum() > 0:
        dil_water = binary_dilation(water_mask, structure=struct)
        boundary  = (flood_mask & dil_water).sum()
        boundary_pct = boundary / max(flood_mask.sum(), 1) * 100
    else:
        boundary_pct = 0.0

    # Difficulty score:
    # High if: flood exists + waterbody exists + they touch a lot
    difficulty = (flood_pct > 0) * 1 + (water_pct > 0) * 1 + (boundary_pct / 100) * 3

    split = 'train' if pid in train_ids else 'val'
    diff_rows.append({
        'patch_id': pid,
        'split': split,
        'flood_pct': round(flood_pct, 2),
        'water_pct': round(water_pct, 2),
        'boundary_pct': round(boundary_pct, 2),
        'difficulty': round(difficulty, 3)
    })

diff_df = pd.DataFrame(diff_rows).sort_values('difficulty', ascending=False)

print('TOP 15 HARDEST PATCHES (model should focus here):')
print(diff_df.head(15)[['patch_id','split','flood_pct','water_pct',
                          'boundary_pct','difficulty']].to_string(index=False))

print(f'\nEASY patches (no flood OR no waterbody): '
      f'{len(diff_df[diff_df["difficulty"] <= 1])} / {len(diff_df)}')
print(f'HARD patches (both present + touching) : '
      f'{len(diff_df[diff_df["boundary_pct"] > 10])} / {len(diff_df)}')

In [ ]:
# ── Difficulty score plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Patch Difficulty Analysis', fontsize=13, fontweight='bold')

# Left: difficulty score bar chart
ax = axes[0]
colors = ['#e63946' if s == 'train' else '#457b9d' for s in diff_df['split']]
ax.bar(range(len(diff_df)), diff_df['difficulty'], color=colors, alpha=0.8)
ax.axhline(2.0, color='orange', linestyle='--', label='Hard threshold')
ax.set_xlabel('Patches (sorted by difficulty)')
ax.set_ylabel('Difficulty Score')
ax.set_title('Difficulty Score per Patch')
legend_elems = [
    mpatches.Patch(color='#e63946', label='Train'),
    mpatches.Patch(color='#457b9d', label='Val'),
]
ax.legend(handles=legend_elems)
ax.grid(axis='y', alpha=0.3)

# Right: flood% vs waterbody% coloured by boundary %
ax = axes[1]
sc = ax.scatter(
    diff_df['flood_pct'], diff_df['water_pct'],
    c=diff_df['boundary_pct'], cmap='RdYlGn_r',
    s=100, alpha=0.85, edgecolors='white', vmin=0, vmax=100
)
plt.colorbar(sc, ax=ax, label='% flood pixels touching WaterBody')
# Mark train vs val
for _, row in diff_df.iterrows():
    marker = 'o' if row['split'] == 'train' else 'D'
ax.set_xlabel('Flood %')
ax.set_ylabel('WaterBody %')
ax.set_title('Difficulty: Red = flood & waterbody tightly mixed')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('difficulty_map.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FINAL SUMMARY — everything in one place
# ═══════════════════════════════════════════════════════════════════
hh_ks_flood_wb, _ = ks_2samp(sar_hh_by_class[1], sar_hh_by_class[2])
hv_ks_flood_wb, _ = ks_2samp(sar_hv_by_class[1], sar_hv_by_class[2])
tex_ks, _         = ks_2samp(texture_by_class[1]['hh_std7'], texture_by_class[2]['hh_std7'])

print('=' * 70)
print('  COMPLETE FINDINGS — SEND THIS TO STRATEGY DISCUSSION')
print('=' * 70)
print(f"""
GAP 1 — SAR SEPARABILITY (Flood vs WaterBody)
  SAR HH KS stat : {hh_ks_flood_wb:.4f}  {'✅ separable' if hh_ks_flood_wb>0.1 else '❌ NOT separable spectrally'}
  SAR HV KS stat : {hv_ks_flood_wb:.4f}  {'✅ separable' if hv_ks_flood_wb>0.1 else '❌ NOT separable spectrally'}
  Texture HH7x7  : {tex_ks:.4f}  {'✅ texture helps!' if tex_ks>0.1 else '⚠️ texture marginally helps' if tex_ks>0.05 else '❌ texture does not help'}

GAP 2 — TRAIN/VAL SPLIT QUALITY
  Train patches : {len(train_df)} | Val patches: {len(val_df)}
  Train flood%  : mean={train_df['Flood_pct'].mean():.1f}%  max={train_df['Flood_pct'].max():.1f}%
  Val   flood%  : mean={val_df['Flood_pct'].mean():.1f}%  max={val_df['Flood_pct'].max():.1f}%
  Train WB%     : mean={train_df['WaterBody_pct'].mean():.1f}%  max={train_df['WaterBody_pct'].max():.1f}%
  Val   WB%     : mean={val_df['WaterBody_pct'].mean():.1f}%  max={val_df['WaterBody_pct'].max():.1f}%

GAP 3 — SPATIAL ADJACENCY
  Patches with both flood AND waterbody : {len(both)} / {len(adj_df)}
  Avg % flood pixels touching WB border : {both['flood_adj_to_water_pct'].mean():.1f}%
  Avg % WB pixels touching flood border : {both['water_adj_to_flood_pct'].mean():.1f}%

DIFFICULTY
  Hard patches (boundary > 10%)  : {len(diff_df[diff_df['boundary_pct']>10])}
  Easy patches (no mixing)       : {len(diff_df[diff_df['difficulty']<=1])}
""")
print('=' * 70)
print('Outputs: sar_separability.png, train_val_split.png, patch_scatter.png,')
print('         boundary_analysis.png, sar_texture.png, difficulty_map.png')